# Ogham OCR: Large Model Comparison

Evaluate large multimodal models (GPT-4o, Gemini, Claude) on real Ogham stone images.

**Goal:** Compare zero-shot and few-shot performance of commercial LLMs against our fine-tuned TrOCR models.

**No GPU needed** -- this notebook uses API calls only.

**Test set:** 21 stones (29 images) from the held-out test split.

**Results saved to:** Google Drive (`ogham_ocr/experiments/large_model_comparison.json`)

---
## 1. Setup

In [ ]:
!pip install -q openai google-generativeai anthropic editdistance Pillow

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ogham_ocr'
os.makedirs(f'{DRIVE_ROOT}/experiments', exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# Clone repository
import os
REPO_DIR = '/content/ogham_ocr'

if os.path.exists(REPO_DIR):
    print('Repository already cloned, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/newsyd04/ogham-masters.git {REPO_DIR}

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

---
## 2. API Keys

Enter your API keys below. They are stored in memory only (never saved to disk or notebook).

**Get keys from:**
- OpenAI: https://platform.openai.com/api-keys
- Google AI: https://aistudio.google.com/app/apikey
- Anthropic: https://console.anthropic.com/settings/keys

In [ ]:
import getpass

OPENAI_API_KEY = getpass.getpass('OpenAI API Key: ')
GOOGLE_API_KEY = getpass.getpass('Google AI API Key: ')
ANTHROPIC_API_KEY = getpass.getpass('Anthropic API Key: ')

print('Keys loaded (not displayed for security)')

---
## 3. Load Test Data

In [ ]:
import json
from pathlib import Path
from PIL import Image
import base64
import io

# Paths -- real data on Drive
# Check multiple possible locations for the data
for candidate in [
    Path(f'{DRIVE_ROOT}/datasets'),        # datasets/splits + datasets/processed
    Path(f'{DRIVE_ROOT}/datasets/real'),    # datasets/real/splits + datasets/real/processed
    Path(f'{DRIVE_ROOT}/ogham_dataset'),    # ogham_dataset/splits + ogham_dataset/processed
    Path(f'{REPO_DIR}/ogham_dataset'),      # repo fallback
]:
    if (candidate / 'splits' / 'test_stones.txt').exists():
        REAL_DATA_DIR = candidate
        break
else:
    raise FileNotFoundError('Could not find test_stones.txt in any expected location')

print(f'Using data from: {REAL_DATA_DIR}')

# Load test stone IDs
with open(REAL_DATA_DIR / 'splits' / 'test_stones.txt') as f:
    test_stone_ids = [line.strip() for line in f if line.strip()]

# Load annotations
with open(REAL_DATA_DIR / 'processed' / 'annotations' / 'transcriptions.json') as f:
    annotations = json.load(f)

# Build test samples: (stone_id, image_path, ground_truth)
test_samples = []
for stone_id in test_stone_ids:
    if stone_id not in annotations:
        continue
    transcription = annotations[stone_id].get('transcription', '')
    if not transcription:
        continue

    crop_dir = REAL_DATA_DIR / 'processed' / 'cropped' / stone_id
    if not crop_dir.exists():
        continue

    image_paths = sorted(crop_dir.glob('*.png'))
    for img_path in image_paths:
        test_samples.append({
            'stone_id': stone_id,
            'image_path': str(img_path),
            'reference': transcription,
        })

print(f'Test samples: {len(test_samples)} images from {len(test_stone_ids)} stones')
for s in test_samples[:5]:
    print(f"  {s['stone_id']}: {s['reference'][:30]}")

In [ ]:
# Few-shot examples (3 train stones with clear, short inscriptions)
FEW_SHOT_STONE_IDS = ['I-CAR-001', 'I-CAR-002', 'I-COR-002']

few_shot_examples = []
for stone_id in FEW_SHOT_STONE_IDS:
    ann = annotations.get(stone_id, {})
    transcription = ann.get('transcription', '')
    crop_dir = REAL_DATA_DIR / 'processed' / 'cropped' / stone_id
    if crop_dir.exists():
        img_path = sorted(crop_dir.glob('*.png'))[0]
        few_shot_examples.append({
            'stone_id': stone_id,
            'image_path': str(img_path),
            'transcription': transcription,
        })

print(f'Few-shot examples: {len(few_shot_examples)}')
for ex in few_shot_examples:
    print(f"  {ex['stone_id']}: {ex['transcription']}")

---
## 4. Helper Functions

In [ ]:
import base64
import io
import time
import editdistance

def image_to_base64(image_path: str) -> str:
    """Convert image to base64 string for API calls."""
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def compute_cer(prediction: str, reference: str) -> float:
    """Compute Character Error Rate between two strings."""
    if len(reference) == 0:
        return 0.0 if len(prediction) == 0 else 1.0
    return editdistance.eval(prediction, reference) / len(reference)

def normalize_prediction(text: str) -> str:
    """Clean up model predictions for fair comparison."""
    # Remove common prefixes models might add
    for prefix in ['The transcription is: ', 'Transcription: ', 'The Ogham text reads: ']:
        if text.startswith(prefix):
            text = text[len(prefix):]
    # Strip whitespace and quotes
    text = text.strip().strip('"').strip("'").strip()
    # Keep only Ogham Unicode characters and spaces
    ogham_chars = set(chr(c) for c in range(0x1680, 0x169D))
    filtered = ''.join(c for c in text if c in ogham_chars or c == ' ')
    return filtered.strip()

def save_results(results: dict, path: str):
    """Save results incrementally to JSON."""
    with open(path, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

RESULTS_PATH = f'{DRIVE_ROOT}/experiments/large_model_comparison.json'

# Load existing results if resuming
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    print(f'Loaded existing results: {list(all_results.keys())}')
else:
    all_results = {}
    print('Starting fresh results')

print(f'Results will be saved to: {RESULTS_PATH}')

In [ ]:
# Prompt templates
ZERO_SHOT_PROMPT = """This is a photograph of an ancient Ogham stone inscription from Ireland (4th-7th century CE).

Ogham is an alphabet of strokes carved along a stone edge. The characters map to Unicode range U+1681 to U+169C.

Please transcribe the Ogham characters visible in this image as Ogham Unicode characters.

IMPORTANT: Return ONLY the Ogham Unicode characters. No explanations, no Latin transliteration, no commentary. Just the raw Ogham text."""

def build_few_shot_prompt(examples):
    """Build few-shot prompt with example images."""
    prompt = """I will show you photographs of ancient Ogham stone inscriptions from Ireland (4th-7th century CE).

Ogham is an alphabet of strokes carved along a stone edge. The characters map to Unicode range U+1681 to U+169C.

Here are some examples of Ogham inscriptions with their correct transcriptions:\n\n"""
    for i, ex in enumerate(examples, 1):
        prompt += f"Example {i}: {ex['transcription']}\n"
    prompt += """\nNow transcribe the following Ogham inscription.

IMPORTANT: Return ONLY the Ogham Unicode characters. No explanations, no Latin transliteration, no commentary. Just the raw Ogham text."""
    return prompt

print('Zero-shot prompt length:', len(ZERO_SHOT_PROMPT), 'chars')
print('Few-shot prompt includes', len(few_shot_examples), 'examples')

---
## 5. GPT-4o Evaluation

In [ ]:
from openai import OpenAI

client_openai = OpenAI(api_key=OPENAI_API_KEY)

def call_gpt4o(image_path: str, prompt: str, few_shot_images=None) -> str:
    """Call GPT-4o with an image and return the response text."""
    messages = []

    # Build content with few-shot images if provided
    if few_shot_images:
        content = [{'type': 'text', 'text': prompt}]
        for ex in few_shot_images:
            img_b64 = image_to_base64(ex['image_path'])
            content.append({
                'type': 'image_url',
                'image_url': {'url': f'data:image/png;base64,{img_b64}', 'detail': 'high'}
            })
            content.append({'type': 'text', 'text': f"Transcription: {ex['transcription']}"})
        content.append({'type': 'text', 'text': '\nNow transcribe this inscription:'})
        img_b64 = image_to_base64(image_path)
        content.append({
            'type': 'image_url',
            'image_url': {'url': f'data:image/png;base64,{img_b64}', 'detail': 'high'}
        })
        messages = [{'role': 'user', 'content': content}]
    else:
        img_b64 = image_to_base64(image_path)
        messages = [{'role': 'user', 'content': [
            {'type': 'text', 'text': prompt},
            {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{img_b64}', 'detail': 'high'}}
        ]}]

    response = client_openai.chat.completions.create(
        model='gpt-4o',
        messages=messages,
        max_tokens=256,
        temperature=0,
    )
    return response.choices[0].message.content

print('GPT-4o client ready')

In [ ]:
# Run GPT-4o zero-shot
print('=== GPT-4o Zero-Shot ===')
gpt4o_zero = {}

for i, sample in enumerate(test_samples):
    stone_id = sample['stone_id']
    img_name = Path(sample['image_path']).name
    key = f"{stone_id}/{img_name}"

    # Skip if already done (resume support)
    if 'gpt4o' in all_results and 'zero_shot' in all_results.get('gpt4o', {}):
        if key in all_results['gpt4o']['zero_shot']:
            print(f'  [{i+1}/{len(test_samples)}] {key} -- cached')
            gpt4o_zero[key] = all_results['gpt4o']['zero_shot'][key]
            continue

    try:
        raw = call_gpt4o(sample['image_path'], ZERO_SHOT_PROMPT)
        pred = normalize_prediction(raw)
        cer = compute_cer(pred, sample['reference'])
        gpt4o_zero[key] = {
            'prediction': pred,
            'raw_response': raw,
            'reference': sample['reference'],
            'cer': cer,
        }
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: CER={cer:.2%} ref='{sample['reference'][:20]}' pred='{pred[:20]}'")
    except Exception as e:
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: ERROR {e}")
        gpt4o_zero[key] = {'error': str(e), 'reference': sample['reference']}

    # Save incrementally
    all_results.setdefault('gpt4o', {})['zero_shot'] = gpt4o_zero
    save_results(all_results, RESULTS_PATH)
    time.sleep(1)  # Rate limit

print(f'\nGPT-4o zero-shot complete: {len(gpt4o_zero)} samples')

In [ ]:
# Run GPT-4o few-shot
print('=== GPT-4o Few-Shot ===')
gpt4o_few = {}

for i, sample in enumerate(test_samples):
    stone_id = sample['stone_id']
    img_name = Path(sample['image_path']).name
    key = f"{stone_id}/{img_name}"

    if 'gpt4o' in all_results and 'few_shot' in all_results.get('gpt4o', {}):
        if key in all_results['gpt4o']['few_shot']:
            print(f'  [{i+1}/{len(test_samples)}] {key} -- cached')
            gpt4o_few[key] = all_results['gpt4o']['few_shot'][key]
            continue

    try:
        prompt = build_few_shot_prompt(few_shot_examples)
        raw = call_gpt4o(sample['image_path'], prompt, few_shot_images=few_shot_examples)
        pred = normalize_prediction(raw)
        cer = compute_cer(pred, sample['reference'])
        gpt4o_few[key] = {
            'prediction': pred,
            'raw_response': raw,
            'reference': sample['reference'],
            'cer': cer,
        }
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: CER={cer:.2%} ref='{sample['reference'][:20]}' pred='{pred[:20]}'")
    except Exception as e:
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: ERROR {e}")
        gpt4o_few[key] = {'error': str(e), 'reference': sample['reference']}

    all_results.setdefault('gpt4o', {})['few_shot'] = gpt4o_few
    save_results(all_results, RESULTS_PATH)
    time.sleep(1)

print(f'\nGPT-4o few-shot complete: {len(gpt4o_few)} samples')

---
## 6. Gemini Evaluation

In [ ]:
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.0-flash')

def call_gemini(image_path: str, prompt: str, few_shot_images=None) -> str:
    """Call Gemini with an image and return the response text."""
    img = Image.open(image_path)

    if few_shot_images:
        content = [prompt]
        for ex in few_shot_images:
            content.append(Image.open(ex['image_path']))
            content.append(f"Transcription: {ex['transcription']}")
        content.append('\nNow transcribe this inscription:')
        content.append(img)
    else:
        content = [prompt, img]

    response = gemini_model.generate_content(
        content,
        generation_config=genai.types.GenerationConfig(
            temperature=0,
            max_output_tokens=256,
        )
    )
    return response.text

print('Gemini client ready')

In [ ]:
# Run Gemini zero-shot
print('=== Gemini Zero-Shot ===')
gemini_zero = {}

for i, sample in enumerate(test_samples):
    stone_id = sample['stone_id']
    img_name = Path(sample['image_path']).name
    key = f"{stone_id}/{img_name}"

    if 'gemini' in all_results and 'zero_shot' in all_results.get('gemini', {}):
        if key in all_results['gemini']['zero_shot']:
            print(f'  [{i+1}/{len(test_samples)}] {key} -- cached')
            gemini_zero[key] = all_results['gemini']['zero_shot'][key]
            continue

    try:
        raw = call_gemini(sample['image_path'], ZERO_SHOT_PROMPT)
        pred = normalize_prediction(raw)
        cer = compute_cer(pred, sample['reference'])
        gemini_zero[key] = {
            'prediction': pred,
            'raw_response': raw,
            'reference': sample['reference'],
            'cer': cer,
        }
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: CER={cer:.2%} ref='{sample['reference'][:20]}' pred='{pred[:20]}'")
    except Exception as e:
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: ERROR {e}")
        gemini_zero[key] = {'error': str(e), 'reference': sample['reference']}

    all_results.setdefault('gemini', {})['zero_shot'] = gemini_zero
    save_results(all_results, RESULTS_PATH)
    time.sleep(0.5)

print(f'\nGemini zero-shot complete: {len(gemini_zero)} samples')

In [ ]:
# Run Gemini few-shot
print('=== Gemini Few-Shot ===')
gemini_few = {}

for i, sample in enumerate(test_samples):
    stone_id = sample['stone_id']
    img_name = Path(sample['image_path']).name
    key = f"{stone_id}/{img_name}"

    if 'gemini' in all_results and 'few_shot' in all_results.get('gemini', {}):
        if key in all_results['gemini']['few_shot']:
            print(f'  [{i+1}/{len(test_samples)}] {key} -- cached')
            gemini_few[key] = all_results['gemini']['few_shot'][key]
            continue

    try:
        prompt = build_few_shot_prompt(few_shot_examples)
        raw = call_gemini(sample['image_path'], prompt, few_shot_images=few_shot_examples)
        pred = normalize_prediction(raw)
        cer = compute_cer(pred, sample['reference'])
        gemini_few[key] = {
            'prediction': pred,
            'raw_response': raw,
            'reference': sample['reference'],
            'cer': cer,
        }
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: CER={cer:.2%} ref='{sample['reference'][:20]}' pred='{pred[:20]}'")
    except Exception as e:
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: ERROR {e}")
        gemini_few[key] = {'error': str(e), 'reference': sample['reference']}

    all_results.setdefault('gemini', {})['few_shot'] = gemini_few
    save_results(all_results, RESULTS_PATH)
    time.sleep(0.5)

print(f'\nGemini few-shot complete: {len(gemini_few)} samples')

---
## 7. Claude Evaluation

In [ ]:
import anthropic

client_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def call_claude(image_path: str, prompt: str, few_shot_images=None) -> str:
    """Call Claude with an image and return the response text."""
    content = []

    if few_shot_images:
        content.append({'type': 'text', 'text': prompt})
        for ex in few_shot_images:
            img_b64 = image_to_base64(ex['image_path'])
            content.append({
                'type': 'image',
                'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64}
            })
            content.append({'type': 'text', 'text': f"Transcription: {ex['transcription']}"})
        content.append({'type': 'text', 'text': '\nNow transcribe this inscription:'})
        img_b64 = image_to_base64(image_path)
        content.append({
            'type': 'image',
            'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64}
        })
    else:
        img_b64 = image_to_base64(image_path)
        content = [
            {'type': 'text', 'text': prompt},
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64}}
        ]

    response = client_anthropic.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=256,
        messages=[{'role': 'user', 'content': content}],
    )
    return response.content[0].text

print('Claude client ready')

In [ ]:
# Run Claude zero-shot
print('=== Claude Zero-Shot ===')
claude_zero = {}

for i, sample in enumerate(test_samples):
    stone_id = sample['stone_id']
    img_name = Path(sample['image_path']).name
    key = f"{stone_id}/{img_name}"

    if 'claude' in all_results and 'zero_shot' in all_results.get('claude', {}):
        if key in all_results['claude']['zero_shot']:
            print(f'  [{i+1}/{len(test_samples)}] {key} -- cached')
            claude_zero[key] = all_results['claude']['zero_shot'][key]
            continue

    try:
        raw = call_claude(sample['image_path'], ZERO_SHOT_PROMPT)
        pred = normalize_prediction(raw)
        cer = compute_cer(pred, sample['reference'])
        claude_zero[key] = {
            'prediction': pred,
            'raw_response': raw,
            'reference': sample['reference'],
            'cer': cer,
        }
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: CER={cer:.2%} ref='{sample['reference'][:20]}' pred='{pred[:20]}'")
    except Exception as e:
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: ERROR {e}")
        claude_zero[key] = {'error': str(e), 'reference': sample['reference']}

    all_results.setdefault('claude', {})['zero_shot'] = claude_zero
    save_results(all_results, RESULTS_PATH)
    time.sleep(1)

print(f'\nClaude zero-shot complete: {len(claude_zero)} samples')

In [ ]:
# Run Claude few-shot
print('=== Claude Few-Shot ===')
claude_few = {}

for i, sample in enumerate(test_samples):
    stone_id = sample['stone_id']
    img_name = Path(sample['image_path']).name
    key = f"{stone_id}/{img_name}"

    if 'claude' in all_results and 'few_shot' in all_results.get('claude', {}):
        if key in all_results['claude']['few_shot']:
            print(f'  [{i+1}/{len(test_samples)}] {key} -- cached')
            claude_few[key] = all_results['claude']['few_shot'][key]
            continue

    try:
        prompt = build_few_shot_prompt(few_shot_examples)
        raw = call_claude(sample['image_path'], prompt, few_shot_images=few_shot_examples)
        pred = normalize_prediction(raw)
        cer = compute_cer(pred, sample['reference'])
        claude_few[key] = {
            'prediction': pred,
            'raw_response': raw,
            'reference': sample['reference'],
            'cer': cer,
        }
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: CER={cer:.2%} ref='{sample['reference'][:20]}' pred='{pred[:20]}'")
    except Exception as e:
        print(f"  [{i+1}/{len(test_samples)}] {stone_id}: ERROR {e}")
        claude_few[key] = {'error': str(e), 'reference': sample['reference']}

    all_results.setdefault('claude', {})['few_shot'] = claude_few
    save_results(all_results, RESULTS_PATH)
    time.sleep(1)

print(f'\nClaude few-shot complete: {len(claude_few)} samples')

---
## 8. Results Summary

In [ ]:
# Compute aggregate metrics for each model x strategy
def summarize(results_dict):
    """Compute mean CER and exact match from a results dict."""
    cers = []
    exact = 0
    errors = 0
    for key, val in results_dict.items():
        if 'error' in val:
            errors += 1
            continue
        cers.append(val['cer'])
        if val['prediction'].strip() == val['reference'].strip():
            exact += 1
    n = len(cers)
    return {
        'mean_cer': sum(cers) / n if n > 0 else None,
        'exact_match': exact / n if n > 0 else None,
        'n_evaluated': n,
        'n_errors': errors,
    }

print(f"{'Model':<20} {'Strategy':<12} {'Mean CER':>10} {'Exact Match':>12} {'N':>5}")
print('-' * 65)

for model_name in ['gpt4o', 'gemini', 'claude']:
    if model_name not in all_results:
        continue
    for strategy in ['zero_shot', 'few_shot']:
        if strategy not in all_results[model_name]:
            continue
        stats = summarize(all_results[model_name][strategy])
        if stats['mean_cer'] is not None:
            print(f"{model_name:<20} {strategy:<12} {stats['mean_cer']:>9.2%} {stats['exact_match']:>11.1%} {stats['n_evaluated']:>5}")
        else:
            print(f"{model_name:<20} {strategy:<12} {'N/A':>10} {'N/A':>12} {stats['n_evaluated']:>5}")

# Add TrOCR reference results
print('-' * 65)
print(f"{'TrOCR-small (synth)':<20} {'fine-tuned':<12} {'0.06%':>10} {'99.8%':>12} {'5000':>5}")
print(f"\nNote: TrOCR results are on synthetic val set, not real test stones.")
print(f"Large model results are on {len(test_samples)} real stone images.")

In [ ]:
# Save final summary to results
summary = {}
for model_name in ['gpt4o', 'gemini', 'claude']:
    if model_name not in all_results:
        continue
    summary[model_name] = {}
    for strategy in ['zero_shot', 'few_shot']:
        if strategy not in all_results[model_name]:
            continue
        summary[model_name][strategy] = summarize(all_results[model_name][strategy])

all_results['summary'] = summary
save_results(all_results, RESULTS_PATH)
print(f'Final results saved to: {RESULTS_PATH}')

In [ ]:
# Per-stone breakdown for the best model
print('\n=== Per-Stone Results (all models, zero-shot) ===\n')
print(f"{'Stone ID':<15} {'Reference':<25} {'GPT-4o':>8} {'Gemini':>8} {'Claude':>8}")
print('-' * 70)

for sample in test_samples:
    stone_id = sample['stone_id']
    img_name = Path(sample['image_path']).name
    key = f"{stone_id}/{img_name}"
    ref = sample['reference'][:22]

    cers = []
    for model_name in ['gpt4o', 'gemini', 'claude']:
        if model_name in all_results and 'zero_shot' in all_results[model_name]:
            val = all_results[model_name]['zero_shot'].get(key, {})
            cer = val.get('cer', None)
            cers.append(f"{cer:.0%}" if cer is not None else 'N/A')
        else:
            cers.append('N/A')

    print(f"{stone_id:<15} {ref:<25} {cers[0]:>8} {cers[1]:>8} {cers[2]:>8}")

---
## 9. Save Markdown Report

In [ ]:
# Generate markdown report
report_lines = ['# Large Model Comparison — Ogham OCR\n']
report_lines.append(f'Date: {time.strftime("%Y-%m-%d %H:%M")}\n')
report_lines.append(f'Test set: {len(test_samples)} images from {len(test_stone_ids)} stones\n\n')

report_lines.append('## Summary\n\n')
report_lines.append(f"| Model | Strategy | Mean CER | Exact Match | N |\n")
report_lines.append(f"|-------|----------|----------|-------------|---|\n")

for model_name in ['gpt4o', 'gemini', 'claude']:
    if model_name not in all_results or 'summary' not in all_results:
        continue
    for strategy in ['zero_shot', 'few_shot']:
        s = all_results.get('summary', {}).get(model_name, {}).get(strategy, {})
        if s.get('mean_cer') is not None:
            report_lines.append(
                f"| {model_name} | {strategy} | {s['mean_cer']:.2%} | {s['exact_match']:.1%} | {s['n_evaluated']} |\n"
            )

report_lines.append(f"| TrOCR-small (synth) | fine-tuned | 0.06% | 99.8% | 5000 |\n\n")
report_lines.append('*TrOCR results are on synthetic validation set. Large model results are on real test stones.*\n')

# Save report
report_path = f'{DRIVE_ROOT}/experiments/large_model_comparison.md'
with open(report_path, 'w') as f:
    f.writelines(report_lines)

print(f'Markdown report saved to: {report_path}')
print('\n' + ''.join(report_lines))